# Interactive Visualization: PDE Datasets

This notebook demonstrates interactive visualization of PDEForge datasets with two examples:

1. **Part 1: Cylinder Flow (2D, Steady-State)** — Flow around an obstacle
2. **Part 2: Burgers Equation (1D, Time-Dependent)** — Shock formation with time slider

---

## Part 1: Cylinder Flow

**Requirements for cylinder flow**: FEniCSx (optional — demo data provided if unavailable)
```bash
conda install -c conda-forge fenics-dolfinx mpich pyvista
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable

import sys
sys.path.insert(0, '..')

# Check for ipywidgets
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, IntSlider, FloatSlider, Dropdown, Checkbox
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("Warning: ipywidgets not found. Install with: pip install ipywidgets")

# Check for FEniCSx
try:
    from pdeforge import generate_dataset, list_models, describe_model
    if 'cylinder_flow_2d' in list_models():
        HAS_FENICSX = True
        print("FEniCSx available. Cylinder flow model ready.")
    else:
        HAS_FENICSX = False
        print("FEniCSx not available. Will use pre-generated data or demo mode.")
except Exception as e:
    HAS_FENICSX = False
    print(f"Import error: {e}")

## 1. Generate Cylinder Flow Dataset

We'll generate a modest number of samples varying the inlet velocity.

In [ ]:
if HAS_FENICSX:
    print(describe_model("cylinder_flow_2d"))

In [ ]:
# Configuration
N_SAMPLES = 12  # Reasonable number for visualization
RESOLUTION = {"x": 110, "y": 41}  # Match channel aspect ratio (2.2 x 0.41)

# Cylinder geometry (for visualization)
CYLINDER_CENTER = (0.2, 0.2)
CYLINDER_RADIUS = 0.05
CHANNEL_LENGTH = 2.2
CHANNEL_HEIGHT = 0.41

In [ ]:
if HAS_FENICSX:
    print(f"Generating {N_SAMPLES} cylinder flow samples...")
    print("(This may take a few minutes due to FEM mesh generation and solving)")
    
    dataset = generate_dataset(
        model="cylinder_flow_2d",
        n_samples=N_SAMPLES,
        resolution=RESOLUTION,
        params={
            "inlet_velocity": 0.3,
            "viscosity": 0.001,
            "cylinder_radius": CYLINDER_RADIUS,
            "cylinder_center": CYLINDER_CENTER,
        },
        seed=42,
    )
    
    print(f"\nGenerated dataset:")
    print(f"  Inputs shape:  {dataset.inputs.shape}")
    print(f"  Outputs shape: {dataset.outputs.shape}")
    print(f"  Input names:   {dataset.input_names}")
    print(f"  Output names:  {dataset.output_names}")
else:
    # Create synthetic demo data
    print("Creating synthetic demo data (FEniCSx not available)")
    
    nx, ny = RESOLUTION["x"], RESOLUTION["y"]
    x = np.linspace(0, CHANNEL_LENGTH, nx)
    y = np.linspace(0, CHANNEL_HEIGHT, ny)
    X, Y = np.meshgrid(x, y)
    
    # Create demo velocity field (parabolic profile with wake)
    def create_demo_sample(inlet_scale):
        U_max = 0.45 * inlet_scale
        u = 4 * U_max * Y * (CHANNEL_HEIGHT - Y) / CHANNEL_HEIGHT**2
        v = np.zeros_like(u)
        
        # Add wake behind cylinder
        wake_x = X - CYLINDER_CENTER[0]
        wake_y = Y - CYLINDER_CENTER[1]
        wake_r = np.sqrt(wake_x**2 + wake_y**2)
        
        # Reduce velocity near and behind cylinder
        wake_mask = (wake_x > 0) & (np.abs(wake_y) < 0.1)
        wake_strength = np.exp(-wake_x / 0.3) * np.exp(-wake_y**2 / 0.01)
        u = u * (1 - 0.7 * wake_strength * (wake_x > 0).astype(float))
        
        # Zero inside cylinder
        inside = wake_r < CYLINDER_RADIUS
        u[inside] = 0
        v[inside] = 0
        
        # Pressure (higher upstream, lower downstream)
        p = -0.5 * (X - 1.1) + 0.1 * np.sin(5 * Y)
        p[inside] = 0
        
        return np.stack([u, v, p], axis=-1)
    
    # Generate samples with varying inlet velocity
    inlet_scales = np.linspace(0.5, 2.0, N_SAMPLES)
    outputs = np.stack([create_demo_sample(s) for s in inlet_scales])
    inputs = inlet_scales.reshape(-1, 1)
    
    # Create simple dataset-like object
    class DemoDataset:
        def __init__(self):
            self.inputs = inputs
            self.outputs = outputs
            self.n_samples = N_SAMPLES
            self.input_names = ["inlet_velocity_scale"]
            self.output_names = ["u", "v", "p"]
            self.grid = {"x": x, "y": y}
            self.metadata = {}
    
    dataset = DemoDataset()
    print(f"Demo dataset created with {N_SAMPLES} samples")

## 2. Visualization Utilities

Custom functions for visualizing flow fields with the cylinder obstacle.

In [ ]:
def add_cylinder(ax, center=CYLINDER_CENTER, radius=CYLINDER_RADIUS):
    """Add cylinder patch to axes."""
    circle = Circle(center, radius, color='gray', ec='black', lw=2, zorder=10)
    ax.add_patch(circle)


def plot_velocity_magnitude(ax, u, v, x, y, title="Velocity Magnitude", cmap='viridis'):
    """Plot velocity magnitude with cylinder."""
    vmag = np.sqrt(u**2 + v**2)
    
    im = ax.contourf(x, y, vmag, levels=30, cmap=cmap)
    add_cylinder(ax)
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(0, CHANNEL_LENGTH)
    ax.set_ylim(0, CHANNEL_HEIGHT)
    
    return im


def plot_streamlines(ax, u, v, x, y, title="Streamlines", density=1.5):
    """Plot streamlines with cylinder."""
    vmag = np.sqrt(u**2 + v**2)
    
    # Background: velocity magnitude
    im = ax.contourf(x, y, vmag, levels=30, cmap='viridis', alpha=0.7)
    
    # Streamlines
    ax.streamplot(x, y, u, v, color='white', density=density, linewidth=0.8, arrowsize=0.8)
    
    add_cylinder(ax)
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(0, CHANNEL_LENGTH)
    ax.set_ylim(0, CHANNEL_HEIGHT)
    
    return im


def plot_pressure(ax, p, x, y, title="Pressure", cmap='RdBu_r'):
    """Plot pressure field with cylinder."""
    vmax = np.abs(p).max()
    if vmax < 1e-10:
        vmax = 1.0
    
    im = ax.contourf(x, y, p, levels=30, cmap=cmap, vmin=-vmax, vmax=vmax)
    add_cylinder(ax)
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(0, CHANNEL_LENGTH)
    ax.set_ylim(0, CHANNEL_HEIGHT)
    
    return im


def plot_velocity_components(ax_u, ax_v, u, v, x, y):
    """Plot u and v velocity components."""
    vmax_u = np.abs(u).max()
    vmax_v = np.abs(v).max()
    if vmax_u < 1e-10: vmax_u = 1.0
    if vmax_v < 1e-10: vmax_v = 1.0
    
    im_u = ax_u.contourf(x, y, u, levels=30, cmap='RdBu_r', vmin=-vmax_u, vmax=vmax_u)
    add_cylinder(ax_u)
    ax_u.set_title('u (x-velocity)')
    ax_u.set_aspect('equal')
    ax_u.set_xlim(0, CHANNEL_LENGTH)
    ax_u.set_ylim(0, CHANNEL_HEIGHT)
    
    im_v = ax_v.contourf(x, y, v, levels=30, cmap='RdBu_r', vmin=-vmax_v, vmax=vmax_v)
    add_cylinder(ax_v)
    ax_v.set_title('v (y-velocity)')
    ax_v.set_aspect('equal')
    ax_v.set_xlim(0, CHANNEL_LENGTH)
    ax_v.set_ylim(0, CHANNEL_HEIGHT)
    
    return im_u, im_v

## 3. Static Visualization: Overview

First, let's look at a single sample in detail.

In [ ]:
# Get grid coordinates
x = dataset.grid['x']
y = dataset.grid['y']

# Get first sample
sample_idx = 0
output = dataset.outputs[sample_idx]
u = output[:, :, 0]
v = output[:, :, 1]
p = output[:, :, 2]
inlet_scale = float(dataset.inputs[sample_idx].flatten()[0])

# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Velocity magnitude
im1 = plot_velocity_magnitude(axes[0, 0], u, v, x, y, "Velocity Magnitude |u|")
plt.colorbar(im1, ax=axes[0, 0], label='m/s')

# Streamlines
im2 = plot_streamlines(axes[0, 1], u, v, x, y, "Streamlines")
plt.colorbar(im2, ax=axes[0, 1], label='|u| (m/s)')

# Pressure
im3 = plot_pressure(axes[1, 0], p, x, y, "Pressure")
plt.colorbar(im3, ax=axes[1, 0], label='Pa')

# u-velocity
vmax = np.abs(u).max()
im4 = axes[1, 1].contourf(x, y, u, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
add_cylinder(axes[1, 1])
axes[1, 1].set_title('u-velocity (horizontal)')
axes[1, 1].set_aspect('equal')
axes[1, 1].set_xlim(0, CHANNEL_LENGTH)
axes[1, 1].set_ylim(0, CHANNEL_HEIGHT)
plt.colorbar(im4, ax=axes[1, 1], label='m/s')

fig.suptitle(f'Sample {sample_idx + 1}: Inlet Scale = {inlet_scale:.2f}', fontsize=14)
plt.tight_layout()
plt.show()

# Print statistics
print(f"\nStatistics for Sample {sample_idx + 1}:")
print(f"  Inlet velocity scale: {inlet_scale:.3f}")
print(f"  Max velocity: {np.sqrt(u**2 + v**2).max():.4f} m/s")
print(f"  Pressure range: [{p.min():.4f}, {p.max():.4f}] Pa")

## 4. Interactive Explorer

Use sliders to browse through all samples and select visualization options.

In [ ]:
if HAS_WIDGETS:
    def explore_sample(
        sample_idx, 
        view_type='Velocity Magnitude', 
        show_streamlines=True,
        streamline_density=1.5
    ):
        """Interactive exploration of a single sample."""
        output = dataset.outputs[sample_idx]
        u = output[:, :, 0]
        v = output[:, :, 1]
        p = output[:, :, 2]
        inlet_scale = float(dataset.inputs[sample_idx].flatten()[0])
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 5))
        
        if view_type == 'Velocity Magnitude':
            vmag = np.sqrt(u**2 + v**2)
            im = ax.contourf(x, y, vmag, levels=30, cmap='viridis')
            plt.colorbar(im, ax=ax, label='|u| (m/s)')
            if show_streamlines:
                ax.streamplot(x, y, u, v, color='white', density=streamline_density, 
                             linewidth=0.7, arrowsize=0.7)
        
        elif view_type == 'u-velocity':
            vmax = np.abs(u).max()
            im = ax.contourf(x, y, u, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            plt.colorbar(im, ax=ax, label='u (m/s)')
        
        elif view_type == 'v-velocity':
            vmax = np.abs(v).max()
            if vmax < 1e-10: vmax = 0.01
            im = ax.contourf(x, y, v, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            plt.colorbar(im, ax=ax, label='v (m/s)')
        
        elif view_type == 'Pressure':
            vmax = np.abs(p).max()
            if vmax < 1e-10: vmax = 1.0
            im = ax.contourf(x, y, p, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            plt.colorbar(im, ax=ax, label='p (Pa)')
        
        elif view_type == 'Vorticity':
            # Compute vorticity: ω = ∂v/∂x - ∂u/∂y
            dx = x[1] - x[0]
            dy = y[1] - y[0]
            dvdx = np.gradient(v, dx, axis=1)
            dudy = np.gradient(u, dy, axis=0)
            omega = dvdx - dudy
            vmax = np.abs(omega).max()
            if vmax < 1e-10: vmax = 1.0
            im = ax.contourf(x, y, omega, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
            plt.colorbar(im, ax=ax, label='ω (1/s)')
        
        add_cylinder(ax)
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(f'{view_type} | Sample {sample_idx + 1}/{dataset.n_samples} | Inlet Scale: {inlet_scale:.2f}')
        ax.set_aspect('equal')
        ax.set_xlim(0, CHANNEL_LENGTH)
        ax.set_ylim(0, CHANNEL_HEIGHT)
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        vmag = np.sqrt(u**2 + v**2)
        print(f"Max velocity: {vmag.max():.4f} m/s")
        print(f"Mean velocity: {vmag.mean():.4f} m/s")
        print(f"Pressure range: [{p.min():.4f}, {p.max():.4f}] Pa")
    
    # Create interactive widget
    sample_slider = IntSlider(
        min=0, max=dataset.n_samples - 1, step=1, value=0,
        description='Sample:', continuous_update=False
    )
    view_dropdown = Dropdown(
        options=['Velocity Magnitude', 'u-velocity', 'v-velocity', 'Pressure', 'Vorticity'],
        value='Velocity Magnitude',
        description='View:'
    )
    streamlines_check = Checkbox(value=True, description='Streamlines')
    density_slider = FloatSlider(
        min=0.5, max=3.0, step=0.25, value=1.5,
        description='Density:', continuous_update=False
    )
    
    interactive_plot = interactive(
        explore_sample,
        sample_idx=sample_slider,
        view_type=view_dropdown,
        show_streamlines=streamlines_check,
        streamline_density=density_slider,
    )
    display(interactive_plot)
else:
    print("Interactive widgets not available. Showing static samples instead.")
    for i in range(min(3, dataset.n_samples)):
        output = dataset.outputs[i]
        u, v, p = output[:,:,0], output[:,:,1], output[:,:,2]
        inlet_scale = float(dataset.inputs[i].flatten()[0])
        
        fig, ax = plt.subplots(figsize=(12, 5))
        vmag = np.sqrt(u**2 + v**2)
        im = ax.contourf(x, y, vmag, levels=30, cmap='viridis')
        ax.streamplot(x, y, u, v, color='white', density=1.5, linewidth=0.7)
        add_cylinder(ax)
        ax.set_title(f'Sample {i+1}: Inlet Scale = {inlet_scale:.2f}')
        ax.set_aspect('equal')
        plt.colorbar(im, ax=ax, label='|u| (m/s)')
        plt.show()

## 5. Compare Across Samples

View multiple samples side-by-side to see the effect of inlet velocity.

In [ ]:
def compare_samples(indices, field='velocity'):
    """Compare multiple samples side by side."""
    n = len(indices)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1:
        axes = [axes]
    
    # Compute global colorbar limits
    if field == 'velocity':
        all_vmag = [np.sqrt(dataset.outputs[i,:,:,0]**2 + dataset.outputs[i,:,:,1]**2) for i in indices]
        vmin, vmax = 0, max(v.max() for v in all_vmag)
    else:
        all_p = [dataset.outputs[i,:,:,2] for i in indices]
        vmax = max(np.abs(v).max() for v in all_p)
        vmin = -vmax
    
    for ax, idx in zip(axes, indices):
        output = dataset.outputs[idx]
        u, v_vel, p = output[:,:,0], output[:,:,1], output[:,:,2]
        inlet_scale = float(dataset.inputs[idx].flatten()[0])
        
        if field == 'velocity':
            vmag = np.sqrt(u**2 + v_vel**2)
            im = ax.contourf(x, y, vmag, levels=30, cmap='viridis', vmin=vmin, vmax=vmax)
            ax.streamplot(x, y, u, v_vel, color='white', density=1.2, linewidth=0.5, arrowsize=0.5)
        else:
            im = ax.contourf(x, y, p, levels=30, cmap='RdBu_r', vmin=vmin, vmax=vmax)
        
        add_cylinder(ax)
        ax.set_title(f'Scale: {inlet_scale:.2f}')
        ax.set_aspect('equal')
        ax.set_xlim(0, CHANNEL_LENGTH)
        ax.set_ylim(0, CHANNEL_HEIGHT)
        ax.set_xlabel('x')
        if ax == axes[0]:
            ax.set_ylabel('y')
    
    # Add colorbar
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    label = '|u| (m/s)' if field == 'velocity' else 'p (Pa)'
    fig.colorbar(im, cax=cbar_ax, label=label)
    
    fig.suptitle(f'{field.capitalize()} Field Comparison', fontsize=14)
    plt.show()

# Compare first, middle, and last samples
indices = [0, dataset.n_samples // 2, dataset.n_samples - 1]
compare_samples(indices, field='velocity')

In [ ]:
# Pressure comparison
compare_samples(indices, field='pressure')

## 6. Profile Plots

Extract 1D profiles at specific locations.

In [ ]:
if HAS_WIDGETS:
    def plot_profiles(sample_idx, x_location, y_location):
        """Plot velocity profiles at specified locations."""
        output = dataset.outputs[sample_idx]
        u = output[:, :, 0]
        v_vel = output[:, :, 1]
        p = output[:, :, 2]
        inlet_scale = float(dataset.inputs[sample_idx].flatten()[0])
        
        # Find nearest indices
        ix = np.argmin(np.abs(x - x_location))
        iy = np.argmin(np.abs(y - y_location))
        
        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        
        # Top row: 2D field with profile lines
        vmag = np.sqrt(u**2 + v_vel**2)
        im = axes[0, 0].contourf(x, y, vmag, levels=30, cmap='viridis')
        axes[0, 0].axvline(x[ix], color='r', linestyle='--', linewidth=2, label=f'x={x[ix]:.2f}')
        axes[0, 0].axhline(y[iy], color='b', linestyle='--', linewidth=2, label=f'y={y[iy]:.2f}')
        add_cylinder(axes[0, 0])
        axes[0, 0].set_title('Velocity Magnitude')
        axes[0, 0].set_aspect('equal')
        axes[0, 0].legend(loc='upper right')
        plt.colorbar(im, ax=axes[0, 0])
        
        # Vertical profile (at x = x_location)
        axes[0, 1].plot(u[:, ix], y, 'b-', linewidth=2, label='u')
        axes[0, 1].plot(v_vel[:, ix], y, 'r-', linewidth=2, label='v')
        axes[0, 1].axhline(CYLINDER_CENTER[1] - CYLINDER_RADIUS, color='gray', linestyle=':', alpha=0.5)
        axes[0, 1].axhline(CYLINDER_CENTER[1] + CYLINDER_RADIUS, color='gray', linestyle=':', alpha=0.5)
        axes[0, 1].set_xlabel('Velocity (m/s)')
        axes[0, 1].set_ylabel('y (m)')
        axes[0, 1].set_title(f'Vertical Profile at x = {x[ix]:.2f}')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Horizontal profile (at y = y_location)
        axes[0, 2].plot(x, u[iy, :], 'b-', linewidth=2, label='u')
        axes[0, 2].plot(x, v_vel[iy, :], 'r-', linewidth=2, label='v')
        axes[0, 2].axvline(CYLINDER_CENTER[0] - CYLINDER_RADIUS, color='gray', linestyle=':', alpha=0.5)
        axes[0, 2].axvline(CYLINDER_CENTER[0] + CYLINDER_RADIUS, color='gray', linestyle=':', alpha=0.5)
        axes[0, 2].set_xlabel('x (m)')
        axes[0, 2].set_ylabel('Velocity (m/s)')
        axes[0, 2].set_title(f'Horizontal Profile at y = {y[iy]:.2f}')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # Bottom row: pressure
        vmax = np.abs(p).max()
        if vmax < 1e-10: vmax = 1.0
        im2 = axes[1, 0].contourf(x, y, p, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        axes[1, 0].axvline(x[ix], color='r', linestyle='--', linewidth=2)
        axes[1, 0].axhline(y[iy], color='b', linestyle='--', linewidth=2)
        add_cylinder(axes[1, 0])
        axes[1, 0].set_title('Pressure')
        axes[1, 0].set_aspect('equal')
        plt.colorbar(im2, ax=axes[1, 0])
        
        axes[1, 1].plot(p[:, ix], y, 'g-', linewidth=2)
        axes[1, 1].set_xlabel('Pressure (Pa)')
        axes[1, 1].set_ylabel('y (m)')
        axes[1, 1].set_title(f'Pressure Profile at x = {x[ix]:.2f}')
        axes[1, 1].grid(True, alpha=0.3)
        
        axes[1, 2].plot(x, p[iy, :], 'g-', linewidth=2)
        axes[1, 2].set_xlabel('x (m)')
        axes[1, 2].set_ylabel('Pressure (Pa)')
        axes[1, 2].set_title(f'Pressure Profile at y = {y[iy]:.2f}')
        axes[1, 2].grid(True, alpha=0.3)
        
        fig.suptitle(f'Sample {sample_idx + 1} | Inlet Scale: {inlet_scale:.2f}', fontsize=14)
        plt.tight_layout()
        plt.show()
    
    # Create interactive widget
    x_slider = FloatSlider(
        min=float(x.min()), max=float(x.max()), step=0.05, value=0.5,
        description='x pos:', continuous_update=False
    )
    y_slider = FloatSlider(
        min=float(y.min()), max=float(y.max()), step=0.02, value=0.2,
        description='y pos:', continuous_update=False
    )
    
    interactive_profiles = interactive(
        plot_profiles,
        sample_idx=IntSlider(min=0, max=dataset.n_samples-1, value=0, description='Sample:', continuous_update=False),
        x_location=x_slider,
        y_location=y_slider,
    )
    display(interactive_profiles)
else:
    print("Interactive widgets not available.")

## 7. Summary Statistics Across Dataset

In [ ]:
# Compute statistics for each sample
inlet_scales = dataset.inputs.flatten()
max_velocities = []
mean_velocities = []
pressure_drops = []

for i in range(dataset.n_samples):
    u = dataset.outputs[i, :, :, 0]
    v = dataset.outputs[i, :, :, 1]
    p = dataset.outputs[i, :, :, 2]
    
    vmag = np.sqrt(u**2 + v**2)
    max_velocities.append(vmag.max())
    mean_velocities.append(vmag.mean())
    
    # Pressure drop (inlet - outlet average)
    p_inlet = p[:, 0].mean()  # Left edge
    p_outlet = p[:, -1].mean()  # Right edge
    pressure_drops.append(p_inlet - p_outlet)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(inlet_scales, max_velocities, c='blue', s=50)
axes[0].set_xlabel('Inlet Scale')
axes[0].set_ylabel('Max Velocity (m/s)')
axes[0].set_title('Max Velocity vs Inlet Scale')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(inlet_scales, mean_velocities, c='green', s=50)
axes[1].set_xlabel('Inlet Scale')
axes[1].set_ylabel('Mean Velocity (m/s)')
axes[1].set_title('Mean Velocity vs Inlet Scale')
axes[1].grid(True, alpha=0.3)

axes[2].scatter(inlet_scales, pressure_drops, c='red', s=50)
axes[2].set_xlabel('Inlet Scale')
axes[2].set_ylabel('Pressure Drop (Pa)')
axes[2].set_title('Pressure Drop vs Inlet Scale')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDataset Statistics:")
print(f"  Inlet scale range: [{inlet_scales.min():.2f}, {inlet_scales.max():.2f}]")
print(f"  Max velocity range: [{min(max_velocities):.4f}, {max(max_velocities):.4f}] m/s")
print(f"  Pressure drop range: [{min(pressure_drops):.4f}, {max(pressure_drops):.4f}] Pa")

---

# Part 2: Time-Dependent Visualization (Burgers 1D)

The cylinder flow above is **steady-state**. Now let's visualize **time-dependent** dynamics using the 1D Burgers equation, which shows shock formation over time.

## 8. Generate Burgers Time Trajectories

We'll modify the Burgers solver to return the full time evolution, not just the final state.

In [ ]:
from pdeforge import get_model

# Get the Burgers model
BurgersModel = get_model("burgers_1d")

# Create model instance with time evolution settings
burgers = BurgersModel(
    resolution={"x": 256},
    viscosity=0.01,      # Moderate viscosity for visible shocks
    time_horizon=1.0,    # Evolve for 1 second
    _n_time_steps=101,   # 101 time steps for smooth animation
)

# Generate several initial conditions and their time trajectories
N_TRAJECTORIES = 6
trajectories = []
initial_conditions = []

print(f"Generating {N_TRAJECTORIES} time trajectories...")

for i in range(N_TRAJECTORIES):
    # Generate random initial condition
    ic = burgers.generate_ic(seed=42 + i)
    initial_conditions.append(ic)
    
    # Solve with full time history
    U_full = burgers.solve(ic, return_full=True)  # Shape: (n_t, nx)
    trajectories.append(U_full)
    
    print(f"  Trajectory {i+1}: IC range [{ic.min():.2f}, {ic.max():.2f}], "
          f"Final range [{U_full[-1].min():.2f}, {U_full[-1].max():.2f}]")

# Get spatial and temporal grids
x_burgers = burgers.grids["x"]
t_burgers = np.linspace(0, burgers.T, burgers.n_t)

print(f"\nGrid: {len(x_burgers)} spatial points, {len(t_burgers)} time steps")
print(f"Time range: [0, {burgers.T}] s")

## 9. Static View: Time Evolution Snapshots

First, let's see snapshots of the solution at different times.

In [ ]:
# Visualize time evolution for one trajectory
traj_idx = 0
U = trajectories[traj_idx]
ic = initial_conditions[traj_idx]

# Select time snapshots
n_snapshots = 6
snapshot_indices = np.linspace(0, len(t_burgers) - 1, n_snapshots, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, t_idx in enumerate(snapshot_indices):
    ax = axes[i]
    
    # Plot solution at this time
    ax.plot(x_burgers, U[t_idx], 'b-', linewidth=2, label=f'u(x, t={t_burgers[t_idx]:.2f})')
    
    # Also show initial condition as reference
    ax.plot(x_burgers, ic, 'k--', linewidth=1, alpha=0.4, label='IC')
    
    ax.set_xlabel('x')
    ax.set_ylabel('u')
    ax.set_title(f't = {t_burgers[t_idx]:.2f} s')
    ax.set_ylim(U.min() - 0.1, U.max() + 0.1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle('Burgers Equation: Shock Formation Over Time', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Space-time diagram (x-t heatmap)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i in range(3):
    U = trajectories[i]
    vmax = np.abs(U).max()
    
    im = axes[i].imshow(
        U.T,  # Transpose so x is vertical, t is horizontal
        aspect='auto',
        origin='lower',
        extent=[0, burgers.T, x_burgers[0], x_burgers[-1]],
        cmap='RdBu_r',
        vmin=-vmax, vmax=vmax
    )
    
    axes[i].set_xlabel('Time t')
    axes[i].set_ylabel('Space x')
    axes[i].set_title(f'Trajectory {i+1}: Space-Time Diagram')
    plt.colorbar(im, ax=axes[i], label='u(x,t)')

plt.tight_layout()
plt.show()

print("Note: Diagonal features show wave propagation; vertical features show shocks (stationary or slow)")

## 10. Interactive Time Slider

Use the slider to scrub through time and watch the solution evolve.

In [ ]:
if HAS_WIDGETS:
    def plot_time_evolution(trajectory_idx, time_idx, show_ic=True, show_spacetime=True):
        """Interactive time evolution viewer."""
        U = trajectories[trajectory_idx]
        ic = initial_conditions[trajectory_idx]
        t = t_burgers[time_idx]
        u = U[time_idx]
        
        if show_spacetime:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            ax1, ax2 = axes
        else:
            fig, ax1 = plt.subplots(1, 1, figsize=(10, 5))
        
        # Left: Solution at current time
        ax1.plot(x_burgers, u, 'b-', linewidth=2.5, label=f'u(x, t={t:.3f})')
        if show_ic:
            ax1.plot(x_burgers, ic, 'k--', linewidth=1.5, alpha=0.5, label='Initial condition')
        
        ax1.set_xlabel('x', fontsize=12)
        ax1.set_ylabel('u', fontsize=12)
        ax1.set_title(f'Burgers Solution at t = {t:.3f} s', fontsize=14)
        ax1.set_ylim(U.min() - 0.2, U.max() + 0.2)
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        
        # Add time indicator
        ax1.text(0.02, 0.98, f'Time: {t:.3f} / {burgers.T:.1f} s', 
                transform=ax1.transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        # Right: Space-time diagram with current time marker
        if show_spacetime:
            vmax = np.abs(U).max()
            im = ax2.imshow(
                U.T,
                aspect='auto',
                origin='lower',
                extent=[0, burgers.T, x_burgers[0], x_burgers[-1]],
                cmap='RdBu_r',
                vmin=-vmax, vmax=vmax
            )
            ax2.axvline(t, color='lime', linewidth=2, linestyle='-', label=f't = {t:.3f}')
            ax2.set_xlabel('Time t', fontsize=12)
            ax2.set_ylabel('Space x', fontsize=12)
            ax2.set_title('Space-Time Diagram', fontsize=14)
            plt.colorbar(im, ax=ax2, label='u(x,t)')
            ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        print(f"Solution statistics at t = {t:.3f}:")
        print(f"  min(u) = {u.min():.4f}, max(u) = {u.max():.4f}, mean(u) = {u.mean():.4f}")
        print(f"  Energy: ∫u² dx = {np.trapz(u**2, x_burgers):.4f}")
    
    # Create widgets
    traj_slider = IntSlider(
        min=0, max=len(trajectories) - 1, step=1, value=0,
        description='Trajectory:', continuous_update=False
    )
    time_slider = IntSlider(
        min=0, max=len(t_burgers) - 1, step=1, value=0,
        description='Time step:', continuous_update=False
    )
    ic_checkbox = Checkbox(value=True, description='Show IC')
    spacetime_checkbox = Checkbox(value=True, description='Show space-time')
    
    interactive_time = interactive(
        plot_time_evolution,
        trajectory_idx=traj_slider,
        time_idx=time_slider,
        show_ic=ic_checkbox,
        show_spacetime=spacetime_checkbox,
    )
    display(interactive_time)
else:
    print("Interactive widgets not available. Showing static animation frames instead.")
    # Show a few frames
    for t_idx in [0, 25, 50, 75, 100]:
        if t_idx < len(t_burgers):
            print(f"\\nt = {t_burgers[t_idx]:.3f}")
            plt.figure(figsize=(8, 4))
            plt.plot(x_burgers, trajectories[0][t_idx], 'b-', linewidth=2)
            plt.plot(x_burgers, initial_conditions[0], 'k--', alpha=0.5)
            plt.xlabel('x')
            plt.ylabel('u')
            plt.title(f't = {t_burgers[t_idx]:.3f}')
            plt.grid(True, alpha=0.3)
            plt.show()

## 11. Play/Pause Animation

Click Play to animate the time evolution automatically.

In [ ]:
if HAS_WIDGETS:
    from IPython.display import display, clear_output
    import time as time_module
    
    # Animation with Play widget
    play = widgets.Play(
        value=0,
        min=0,
        max=len(t_burgers) - 1,
        step=1,
        interval=50,  # milliseconds between frames
        description="Play",
        disabled=False
    )
    
    time_slider_anim = IntSlider(
        min=0, max=len(t_burgers) - 1, step=1, value=0,
        description='Time:',
        continuous_update=True,
        readout=False,
    )
    
    # Link play and slider
    widgets.jslink((play, 'value'), (time_slider_anim, 'value'))
    
    traj_dropdown = Dropdown(
        options=[(f'Trajectory {i+1}', i) for i in range(len(trajectories))],
        value=0,
        description='Sample:'
    )
    
    output = widgets.Output()
    
    def update_animation(change):
        with output:
            clear_output(wait=True)
            
            traj_idx = traj_dropdown.value
            t_idx = time_slider_anim.value
            
            U = trajectories[traj_idx]
            ic = initial_conditions[traj_idx]
            t = t_burgers[t_idx]
            u = U[t_idx]
            
            fig, ax = plt.subplots(1, 1, figsize=(10, 5))
            
            ax.plot(x_burgers, u, 'b-', linewidth=2.5)
            ax.plot(x_burgers, ic, 'k--', linewidth=1, alpha=0.4, label='Initial')
            ax.fill_between(x_burgers, u, alpha=0.3)
            
            ax.set_xlabel('x', fontsize=12)
            ax.set_ylabel('u(x, t)', fontsize=12)
            ax.set_title(f'Burgers Equation: t = {t:.3f} s', fontsize=14)
            ax.set_ylim(U.min() - 0.2, U.max() + 0.2)
            ax.set_xlim(x_burgers[0], x_burgers[-1])
            ax.legend(loc='upper right')
            ax.grid(True, alpha=0.3)
            
            # Progress bar
            progress = t / burgers.T
            ax.axhline(U.min() - 0.15, xmin=0, xmax=progress, color='green', linewidth=4)
            
            plt.tight_layout()
            plt.show()
    
    time_slider_anim.observe(update_animation, names='value')
    traj_dropdown.observe(update_animation, names='value')
    
    # Layout
    controls = widgets.HBox([play, time_slider_anim, traj_dropdown])
    display(widgets.VBox([controls, output]))
    
    # Initial display
    update_animation(None)
else:
    print("Animation requires ipywidgets.")

## 12. Compare Multiple Trajectories

View all trajectories at the same time instant.

In [ ]:
if HAS_WIDGETS:
    def compare_trajectories_at_time(time_idx):
        """Compare all trajectories at the same time."""
        t = t_burgers[time_idx]
        
        n_rows = 2
        n_cols = 3
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 8))
        axes = axes.flatten()
        
        # Get global y-limits
        all_min = min(U.min() for U in trajectories)
        all_max = max(U.max() for U in trajectories)
        
        for i, (U, ic) in enumerate(zip(trajectories, initial_conditions)):
            if i >= len(axes):
                break
            ax = axes[i]
            
            ax.plot(x_burgers, U[time_idx], 'b-', linewidth=2, label=f't = {t:.2f}')
            ax.plot(x_burgers, ic, 'k--', linewidth=1, alpha=0.4, label='IC')
            ax.fill_between(x_burgers, U[time_idx], alpha=0.2)
            
            ax.set_xlabel('x')
            ax.set_ylabel('u')
            ax.set_title(f'Trajectory {i+1}')
            ax.set_ylim(all_min - 0.1, all_max + 0.1)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)
        
        fig.suptitle(f'All Trajectories at t = {t:.3f} s', fontsize=14)
        plt.tight_layout()
        plt.show()
    
    compare_slider = IntSlider(
        min=0, max=len(t_burgers) - 1, step=1, value=len(t_burgers) // 2,
        description='Time step:', continuous_update=False
    )
    
    interactive_compare = interactive(compare_trajectories_at_time, time_idx=compare_slider)
    display(interactive_compare)
else:
    # Static comparison at final time
    compare_trajectories_at_time = lambda t_idx: None  # placeholder
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    
    for i, (U, ic) in enumerate(zip(trajectories[:6], initial_conditions[:6])):
        axes[i].plot(x_burgers, U[-1], 'b-', linewidth=2)
        axes[i].plot(x_burgers, ic, 'k--', alpha=0.4)
        axes[i].set_title(f'Trajectory {i+1} (final)')
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle('All Trajectories at Final Time')
    plt.tight_layout()
    plt.show()

## Summary

This notebook demonstrated visualization for two types of PDEs:

### Part 1: Steady-State (Cylinder Flow)
- 2D flow around a cylinder obstacle
- Visualized: velocity magnitude, streamlines, pressure, vorticity
- Interactive exploration of samples with different inlet velocities

### Part 2: Time-Dependent (Burgers 1D)
- Shock formation and propagation over time
- Space-time diagrams showing wave characteristics
- Interactive time slider and play/pause animation
- Comparison across multiple initial conditions

### For Operator Learning

| Problem | Input | Output | Challenge |
|---------|-------|--------|-----------|
| Cylinder Flow | Inlet velocity (scalar) | (u, v, p) field | Complex geometry |
| Burgers 1D | u(x, t=0) | u(x, t=T) | Shock discontinuities |

Both problems benefit from **uncertainty quantification** — see `03_uq_workflow.ipynb`.